# learn-better — Google Colab setup

One-click setup for the [learn-better](https://github.com/dragosbo/learn-better) YouTube-to-study-material pipeline.

**How to use:** open this notebook in Colab (the *Open In Colab* badge in the repo README), then **Runtime → Run all**. Cells 1–2 install everything; cell 4 verifies; cell 5 does a real download + transcribe. The cells are idempotent — safe to re-run after a disconnect.

> Colab runs Linux, so the Windows `.bat` runners don't apply here — call the tools with `python code/<script>.py` (optionally a `config/*.json` argument). All outputs land under `data/` (see `lib/paths.py`).

## Cell 1 — System deps + clone the repo (run once per session)

In [ ]:
import os

# ffmpeg is a SYSTEM binary (not pip-installable) that yt-dlp/faster-whisper need.
!apt-get install -y ffmpeg -q

REPO_URL = "https://github.com/dragosbo/learn-better.git"
REPO_DIR = "/content/learn-better"

# Idempotent: only clone if it isn't already here.
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}

## Cell 2 — Python dependencies

Colab already ships its own Jupyter stack (`ipykernel`, `notebook`, …). Installing the repo's `ipykernel` pin on top of it triggers harmless-but-noisy `pip` "dependency conflict" warnings and can nag you to restart the runtime. `ipykernel` is only needed to run the notebooks in a *local* dev environment — on Colab it's redundant (Colab **is** the notebook host). So we install everything **except** `ipykernel` here to keep the output clean.

In [ ]:
# Install requirements WITHOUT ipykernel (Colab provides its own Jupyter stack;
# skipping it avoids the noisy 'dependency conflict' warnings + restart prompts).
with open("requirements.txt") as f:
    reqs = [ln for ln in f if ln.strip() and not ln.lstrip().lower().startswith("ipykernel")]
with open("/tmp/requirements.colab.txt", "w") as f:
    f.writelines(reqs)

!pip install -q -r /tmp/requirements.colab.txt
print("\n✅ Dependencies installed (ipykernel skipped — Colab supplies it).")

## Cell 3 (OPTIONAL) — Persist outputs to Google Drive

**You can skip this cell.** The pipeline works fully without it — the only difference is that everything under `data/` lives in the temporary Colab session and is **lost when the runtime disconnects**. Mount Drive if you want your audio / transcripts / summaries to survive between sessions.

If the mount fails (`ValueError: mount failed`), it's almost always the Google auth popup being dismissed, timing out, or blocked by third-party cookies — **not** a problem with this project. The cell below catches that so it won't stop **Run all**; just re-run it and complete the popup (grant all permissions, same Google account). Files still work without Drive — see "Accessing your files" below.

In [ ]:
import os

DRIVE_OUT = None
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT = "/content/drive/MyDrive/learn-better-output"
    # Subdirs mirror lib/paths.py (all outputs live under data/).
    for sub in ["audio", "audio_reencoded", "transcripts", "generated_transcripts",
                "summaries", "tts_output", "wordclouds", "whisper-model-cache"]:
        os.makedirs(f"{DRIVE_OUT}/{sub}", exist_ok=True)
    print(f"✅ Drive mounted. Persist outputs by copying data/ into: {DRIVE_OUT}")
except Exception as e:
    print(f"⚠️  Drive NOT mounted ({type(e).__name__}: {e}).")
    print("   This is optional — continuing without Drive. Outputs stay under")
    print("   /content/learn-better/data and vanish when the session ends.")
    print("   To retry: re-run this cell and complete the Google popup fully.")

## Cell 4 — Verify the environment + check your accelerator (CPU / GPU / TPU)

This confirms ffmpeg + the Python deps, and tells you **which hardware Colab gave you**:
- **GPU** (usually a **T4**) — best for this project; `faster-whisper` uses it automatically and runs Whisper ~5–10× faster.
- **CPU** — the default; everything works, just slower on large Whisper models.
- **TPU** — shown for completeness, but **not useful here** (Whisper/CTranslate2 don't use TPUs; it behaves like CPU for us). Prefer GPU or CPU.

**To change it:** `Runtime → Change runtime type → Hardware accelerator → T4 GPU`, then reconnect and re-run. (Changing it resets the VM — you'll re-run cells 1–2.)

In [ ]:
# --- Which accelerator do I have? ---
import subprocess, os

accel = "CPU"

# GPU? nvidia-smi exists only when a GPU runtime is attached.
try:
    out = subprocess.run(["nvidia-smi",
                          "--query-gpu=name,memory.total",
                          "--format=csv,noheader"],
                         capture_output=True, text=True)
    if out.returncode == 0 and out.stdout.strip():
        accel = "GPU"
        print(f"🚀 GPU detected: {out.stdout.strip()}")
except FileNotFoundError:
    pass

# TPU? Colab exposes it via the COLAB_TPU_ADDR env var (or a TPU runtime).
if accel == "CPU" and (os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME")):
    accel = "TPU"
    print("️  TPU runtime detected — note: this project can't use a TPU")
    print("   (Whisper/CTranslate2 target CPU/GPU). It will run like CPU here;")
    print("   switch to 'T4 GPU' for a real speedup.")

if accel == "CPU":
    print("🖥️  CPU only (no GPU/TPU attached). Everything works; large Whisper")
    print("   models are slower. For a free GPU: Runtime -> Change runtime type -> T4 GPU.")

print(f"\n== Accelerator in use: {accel} ==\n")

# --- Verify the toolchain ---
!ffmpeg -version | head -1
import yt_dlp, faster_whisper, pandas
print('✅ All imports OK — environment ready.')

## Cell 5 — Example run (download → transcribe)

The tools are config-driven and follow a pipeline: **first download** audio + transcripts for a source, **then** transcribe/summarize what you downloaded. A fresh clone has an empty `data/audio/`, so we download first — otherwise the transcribe step correctly reports *"No audio files matched"* (nothing to do yet).

`read_channel.py` reads its source from the config block at the top of the file (`PLAYLIST_ID` / `CHANNEL` / `SEARCH` / `LIMIT`). The cell below points it at a search with `sed`, then runs the download + a Whisper transcription. Edit the search term to whatever you want.

> **English only, on purpose.** On a cloud IP, YouTube often rate-limits subtitle requests (`HTTP Error 429: Too Many Requests`). Fetching **only English** (one request per video instead of `en`+`fr`+`ro`) makes a first run much less likely to trip the limit. We set `languages.json` to `["en"]` for the demo; the next cell shows how to add French later. If you still hit a 429, just wait a minute and re-run — it's transient and skip-if-exists resumes where it left off.

In [ ]:
# 1) Fetch ENGLISH subtitles only (fewer requests -> avoids YouTube's 429 on cloud IPs).
import json
with open("code/languages.json", "w") as f:
    json.dump(["en"], f)
print('languages.json ->', open('code/languages.json').read())

# 2) Point read_channel.py at a search (or set PLAYLIST_ID / CHANNEL instead), keep it small.
!sed -i 's|^SEARCH *=.*|SEARCH = "git tutorial for beginners"|' code/read_channel.py
!sed -i 's|^LIMIT *=.*|LIMIT = 1|' code/read_channel.py

# 3) Download audio + English transcript -> data/audio/ , data/transcripts/
!python code/read_channel.py

# 4) Transcribe the downloaded audio with Whisper -> data/generated_transcripts/
#    (uses config/config_transcribe.json; edit it to target specific files)
!python code/transcribe_audio.py config/config_transcribe.json

## Cell 6 (OPTIONAL) — Add French / other-language subtitles later

Once the English demo works, you can widen the languages. This re-runs the download so YouTube captions in the extra languages are fetched too (audio is already present, so it's skipped). If YouTube returns a `429` for a language, wait a minute and re-run — it's a temporary cloud-IP rate limit, not a failure.

The repo supports `en`, `fr`, `ro` out of the box (see `config/languages.json` usage). Whisper can also **translate** any audio to English or transcribe French/Romanian directly — see the `config/config_transcribe.*.json` variants (`.fr.json`, `.ro.json`, `.translate.json`).

In [ ]:
# Widen to English + French (add "ro" for Romanian). Then re-run the download.
import json
with open("code/languages.json", "w") as f:
    json.dump(["en", "fr"], f)
print('languages.json ->', open('code/languages.json').read())

# Audio already downloaded -> skipped; this just fetches the extra-language captions.
!python code/read_channel.py

# Optional: Whisper-transcribe a French clip in French, or translate any clip to English:
#   !python code/transcribe_audio.py config/config_transcribe.fr.json         # French -> French
#   !python code/transcribe_audio.py config/config_transcribe.translate.json  # any -> English

## Accessing your files

All outputs land under `data/` inside the repo: `data/audio/`, `data/transcripts/`, `data/generated_transcripts/`, `data/summaries/`, `data/tts_output/`, `data/wordclouds/`.

Three ways to get at them:
1. **File browser (easiest):** click the 📁 folder icon in Colab's left sidebar → expand `learn-better/data/` → right-click a file → **Download**.
2. **List them in a cell:** run the cell below.
3. **Download one file to your computer:** `from google.colab import files; files.download('<path>')`.

> **Persistence:** Colab wipes `/content/` when the session ends. To keep files, either download them, or mount Drive (Cell 3) and copy `data/` there (the last cell does this automatically if Drive is mounted).

In [ ]:
# List everything produced under data/
!ls -R /content/learn-better/data 2>/dev/null || echo 'No data/ yet — run the example (Cell 5) first.'

# If Drive was mounted (Cell 3), copy all outputs there so they survive the session.
import os, shutil, pathlib
if 'DRIVE_OUT' in globals() and DRIVE_OUT:
    data = pathlib.Path('/content/learn-better/data')
    if data.is_dir():
        for src in data.rglob('*'):
            if src.is_file():
                dest = pathlib.Path(DRIVE_OUT) / src.relative_to(data)
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy(src, dest)
        print(f'✅ Copied data/ -> {DRIVE_OUT}')
    else:
        print('No data/ to copy yet.')
else:
    print('Drive not mounted — download files from the 📁 sidebar before disconnecting.')

## Finishing up / logging out

When you're done, clean up so you don't keep burning free Colab compute and so no session is left holding your Google account:

1. **Save first.** Download what you need (📁 sidebar) or make sure the copy-to-Drive cell above ran — `/content/` is wiped on disconnect.
2. **Unmount Drive** (if you mounted it) — run the cell below, or just disconnect the runtime (that releases it too).
3. **Disconnect the runtime:** `Runtime → Disconnect and delete runtime`. This frees the VM/GPU and stops counting against your usage. (Closing the browser tab does NOT stop it immediately — the runtime keeps running until it idles out.)
4. **Sign out of Google (shared/public computer only):** Colab uses your Google login; there's no separate Colab logout. Click your **account avatar (top-right) → Sign out**, or sign out of Google in the browser. If you set a `cookies.txt` or a `GITHUB_TOKEN` this session, delete it too (it lived only in this VM, but don't leave copies in Drive).

> Nothing here is permanent to your machine — the whole environment is the throwaway Colab VM. "Logging out" is really just: save → disconnect the runtime → (optionally) sign out of Google on a shared device.

In [ ]:
# Optional cleanup: unmount Drive (if mounted). Then use Runtime -> Disconnect and delete runtime.
try:
    from google.colab import drive
    drive.flush_and_unmount()
    print('✅ Drive flushed and unmounted. Now: Runtime -> Disconnect and delete runtime.')
except Exception as e:
    print(f'Nothing to unmount ({type(e).__name__}). Just: Runtime -> Disconnect and delete runtime.')